In [6]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import torch
import numpy as np
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [39]:
PROJECT_ROOT = Path("/home/ayush/Desktop/tutorials/rl_latent/E2C/")
eig_path = 'runs/coffee/coffee_eig_0'
maxdyn_path = 'runs/coffee/coffee_maxdyn_0'
rand_path = 'runs/coffee/coffee_random_0'

paths = [eig_path, maxdyn_path, rand_path]
titles = ['EIG Pos', 'MaxDyn Pos', 'Random Pos', 'EIG Vel', 'MaxDyn Vel', 'Random Vel']

# Load all states to calculate global limits
states = [torch.load(PROJECT_ROOT / path / 'eval_saved_state.pt') for path in paths]

# states: list of 3 tensors/arrays; cols: [x, y, z, gripper, x_obj, y_obj, z_obj, reward, xdot, ydot, zdot, gripper_vel]
states_np = [s.cpu().numpy() if hasattr(s, "cpu") else np.asarray(s) for s in states]

bins = 10
min_count = 3  # for arrows if you add them later

def hist2d_with_velocity(arr, bins):
    x, y = arr[:, 0], arr[:, 1]
    vx, vy, vz = arr[:, 8], arr[:, 9], arr[:, 10]
    vel = np.linalg.norm((vx, vy, vz), axis=0)
    occ_pos, xedges, yedges = np.histogram2d(x, y, bins=bins)
    occ_vel, _, _ = np.histogram2d(x, y, bins=bins, weights=vel)
    
    # vx_sum, _, _ = np.histogram2d(x, y, bins=[xedges, yedges], weights=vx)
    # vy_sum, _, _ = np.histogram2d(x, y, bins=[xedges, yedges], weights=vy)
    # vz_sum, _, _ = np.histogram2d(x, y, bins=[xedges, yedges], weights=vz)
    # speed_sum, _, _ = np.histogram2d(x, y, bins=[xedges, yedges],
    #                                  weights=np.linalg.norm((vx, vy, vz), axis=0))
    # mean_vx = np.divide(vx_sum, occ, out=np.zeros_like(vx_sum), where=occ > 0)
    # mean_vy = np.divide(vy_sum, occ, out=np.zeros_like(vy_sum), where=occ > 0)
    # mean_speed = np.divide(speed_sum, occ, out=np.zeros_like(speed_sum), where=occ > 0)
    # bin centers (use these for Heatmap)
    xc = 0.5 * (xedges[:-1] + xedges[1:])
    yc = 0.5 * (yedges[:-1] + yedges[1:])
    return occ_pos, occ_vel, xc, yc

# Compute global axis limits so columns align visually
x_min = min(arr[:, 0].min() for arr in states_np)
x_max = max(arr[:, 0].max() for arr in states_np)
y_min = min(arr[:, 1].min() for arr in states_np)
y_max = max(arr[:, 1].max() for arr in states_np)

fig = make_subplots(rows=2, cols=3, subplot_titles=titles,
                    horizontal_spacing=0.05, vertical_spacing=0.15)

for col, arr in enumerate(states_np, start=1):
    occ_pos, occ_vel, xc, yc = hist2d_with_velocity(arr, bins)

    # Heatmaps: use centers for x/y
    fig.add_trace(
        go.Heatmap(x=xc, y=yc, z=occ_pos.T, colorscale="Magma",
                   colorbar=dict(title="count"), showscale=(col == 3)),
        row=1, col=col
    )
    fig.add_trace(
        go.Heatmap(x=xc, y=yc, z=occ_vel.T, colorscale="Magma",
                   colorbar=dict(title="mean|v|"), showscale=(col == 3)),
        row=2, col=col
    )

    # Top-row markers: start (red circle), end (green circle), object (contrast star)
    start_x, start_y = arr[0, 0], arr[0, 1]
    end_x, end_y = arr[-1, 0], arr[-1, 1]
    obj_x, obj_y = arr[0, 4], arr[0, 5]

    fig.add_trace(
        go.Scatter(x=[start_x], y=[start_y], mode="markers",
                   marker=dict(size=10, color="red", symbol="circle"),
                   name=f"start {col-1}", showlegend=(col == 1)),
        row=1, col=col
    )
    fig.add_trace(
        go.Scatter(x=[end_x], y=[end_y], mode="markers",
                   marker=dict(size=10, color="green", symbol="circle"),
                   name=f"end {col-1}", showlegend=(col == 1)),
        row=1, col=col
    )
    fig.add_trace(
        go.Scatter(x=[obj_x], y=[obj_y], mode="markers",
                   marker=dict(size=12, color="#FFD700", symbol="star"),
                   name=f"object {col-1}", showlegend=(col == 1)),
        row=1, col=col
    )

# Axis labels and unified ranges
for c in range(1, 4):
    fig.update_xaxes(title_text="x", range=[x_min, x_max], row=1, col=c)
    fig.update_yaxes(title_text="y", range=[y_min, y_max], row=1, col=c)
    fig.update_xaxes(title_text="x", range=[x_min, x_max], row=2, col=c)
    fig.update_yaxes(title_text="y", range=[y_min, y_max], row=2, col=c)

fig.update_layout(height=650, width=1150,
                  title_text="EE XY occupancy and velocity by run",
                  template="plotly_dark")
fig.show()

In [ ]:
PROJECT_ROOT = Path("/home/ayush/Desktop/tutorials/rl_latent/E2C/")
eig_path = 'runs/coffee/coffee_eig_0'
maxdyn_path = 'runs/coffee/coffee_maxdyn_0'
rand_path = 'runs/coffee/coffee_random_0'

paths = [eig_path, maxdyn_path, rand_path]
titles = ['EIG Pos', 'MaxDyn Pos', 'Random Pos', 'EIG Vel', 'MaxDyn Vel', 'Random Vel']

# Load all states to calculate global limits
states = [torch.load(PROJECT_ROOT / path / 'eval_saved_state.pt') for path in paths]

# states: list/tuple of 3 tensors/arrays; each with cols [x, y, z, gripper, x_obj, y_obj, z_obj, reward, xdot, ydot, zdot, gripper_vel]
# ensure numpy
states_np = []
for s in states:
    s = s.cpu().numpy() if hasattr(s, "cpu") else np.asarray(s)
    states_np.append(s)

bins = 30
min_count = 3  # quiver only where enough samples

def hist2d_with_velocity(arr, bins):
    x, y = arr[:, 0], arr[:, 1]
    vx, vy, vz = arr[:, 8], arr[:, 9], arr[:, 10]
    occ, xedges, yedges = np.histogram2d(x, y, bins=bins)
    vx_sum, _, _ = np.histogram2d(x, y, bins=[xedges, yedges], weights=vx)
    vy_sum, _, _ = np.histogram2d(x, y, bins=[xedges, yedges], weights=vy)
    vz_sum, _, _ = np.histogram2d(x, y, bins=[xedges, yedges], weights=vz)
    speed_sum, _, _ = np.histogram2d(x, y, bins=[xedges, yedges], weights=np.linalg.norm((vx, vy, vz), axis=0))
    mean_vx = np.divide(vx_sum, occ, out=np.zeros_like(vx_sum), where=occ > 0)
    mean_vy = np.divide(vy_sum, occ, out=np.zeros_like(vy_sum), where=occ > 0)
    mean_vz = np.divide(vz_sum, occ, out=np.zeros_like(vz_sum), where=occ > 0)
    mean_speed = np.divide(speed_sum, occ, out=np.zeros_like(speed_sum), where=occ > 0)
    # bin centers for quiver
    xc = 0.5 * (xedges[:-1] + xedges[1:])
    yc = 0.5 * (yedges[:-1] + yedges[1:])
    return occ, mean_speed, mean_vx, mean_vy, mean_vz, xc, yc, xedges, yedges

fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=titles,
    horizontal_spacing=0.05, vertical_spacing=0.15
)

for col, arr in enumerate(states_np, start=1):
    occ, mean_speed, mean_vx, mean_vy, mean_vz, xc, yc, xedges, yedges = hist2d_with_velocity(arr, bins)
    # Heatmaps
    fig.add_trace(
        go.Heatmap(
            x=xedges, y=yedges, z=occ.T,
            colorscale="Magma", colorbar=dict(title="count"),
            showscale=(col == 3)  # show one colorbar on the right of top row
        ),
        row=1, col=col
    )
    fig.add_trace(
        go.Heatmap(
            x=xedges, y=yedges, z=mean_speed.T,
            colorscale="Viridis", colorbar=dict(title="mean|v|"),
            showscale=(col == 3)  # show one colorbar on the right of bottom row
        ),
        row=2, col=col
    )

fig.update_xaxes(title_text="x", row=2, col=1)
fig.update_yaxes(title_text="y", row=1, col=1)
fig.update_layout(height=600, width=1100, title_text="EE XY occupancy and velocity by run", template="plotly_dark")
fig.show()

# Calculate global min/max for X and Y
# # x_min = min(state[:, 0].min() for state in states)
# # x_max = max(state[:, 0].max() for state in states)
# y_min = min(state[:, 1].min() for state in states)
# y_max = max(state[:, 1].max() for state in states)
# x_max = 0.7
# x_min = -0.7
# num_bins = 20

# fig_spatial, axes_spatial = plt.subplots(2, 3, figsize=(15, 10))
# fig_spatial.suptitle('XY Plane EE Coverage Heatmap (Pos and Vel)', fontsize=16)


In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()
run_path = eig_path

# Load saved state
state = torch.load(PROJECT_ROOT / run_path / 'eval_state.pt')
print(f"State shape: {state.shape}")

# Convert to numpy
ee_positions = state[:, 0:3]
if isinstance(ee_positions, torch.Tensor):
    ee_positions = ee_positions.cpu().numpy()

print(f"EE positions shape: {ee_positions.shape}")
print(f"First few positions:\n{ee_positions[:5]}")
print(f"Min/Max values: X({ee_positions[:, 0].min():.3f}, {ee_positions[:, 0].max():.3f}), "
      f"Y({ee_positions[:, 1].min():.3f}, {ee_positions[:, 1].max():.3f}), "
      f"Z({ee_positions[:, 2].min():.3f}, {ee_positions[:, 2].max():.3f})")

x, y, z = ee_positions[:, 0], ee_positions[:, 1], ee_positions[:, 2]

# Create static 3D plot
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Plot trajectory with decreasing alpha
# First plot connecting lines with decreasing alpha
for i in range(len(x) - 1):
    alpha = 1.0 - (i / len(x))  # starts at 1, decreases to ~0
    ax.plot(x[i:i+2], y[i:i+2], z[i:i+2], color='blue', linewidth=1.5, alpha=alpha)

# Then plot points with decreasing alpha
for i in range(len(x)):
    alpha = 1.0 - (i / len(x))  # starts at 1, decreases to ~0
    ax.scatter(x[i], y[i], z[i], color='blue', s=10, alpha=alpha)
    
ax.scatter(x[0], y[0], z[0], color='green', s=100, label='Start', marker='o')
ax.scatter(x[-1], y[-1], z[-1], color='red', s=100, label='End', marker='s')
ax.scatter(state[-1, 6], state[-1, 7], state[-1, 8], color='pink', s=100, label='Button', marker='*')

ax.set_xlim(x.min(), x.max())
ax.set_ylim(y.min(), y.max())
ax.set_zlim(z.min(), z.max())

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.set_title('End-Effector Trajectory (Static)')
ax.legend()
ax.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Animated 3D trajectory
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

fig_anim = plt.figure(figsize=(10, 8))
ax_anim = fig_anim.add_subplot(111, projection='3d')

# Set fixed limits
ax_anim.set_xlim(x.min(), x.max())
ax_anim.set_ylim(y.min(), y.max())
ax_anim.set_zlim(z.min(), z.max())
ax_anim.set_xlabel('X')
ax_anim.set_ylabel('Y')
ax_anim.set_zlabel('Z')
ax_anim.set_title('End-Effector Trajectory (Animated)')
ax_anim.grid(True)

# Plot start and end points
ax_anim.scatter(x[0], y[0], z[0], color='green', s=100, label='Start', marker='o')
ax_anim.scatter(x[-1], y[-1], z[-1], color='red', s=100, label='End', marker='s')
ax_anim.scatter(state[-1, 6], state[-1, 7], state[-1, 8], color='pink', s=100, label='Button', marker='*')

def update_frame(frame):
    ax_anim.clear()
    
    # Redraw fixed elements
    ax_anim.set_xlim(x_min, x_max)
    ax_anim.set_ylim(y.min(), y.max())
    ax_anim.set_zlim(z.min(), z.max())
    ax_anim.set_xlabel('X')
    ax_anim.set_ylabel('Y')
    ax_anim.set_zlabel('Z')
    ax_anim.set_title(f'End-Effector Trajectory (Step {frame}/{len(x)})')
    ax_anim.grid(True)
    
    # Draw trajectory up to current frame with fading alpha
    for i in range(max(0, frame - 20), frame):
        # alpha = 1.0 - (i / len(x))
        alpha = 1.0
        if i < frame - 1:
            ax_anim.plot(x[i:i+2], y[i:i+2], z[i:i+2], color='blue', linewidth=1.5, alpha=alpha)
        ax_anim.scatter(x[i], y[i], z[i], color='blue', s=10, alpha=alpha)
    
    # Current point in bright color
    if frame > 0:
        ax_anim.scatter(x[frame-1], y[frame-1], z[frame-1], color='cyan', s=50, alpha=1.0)
    
    # Fixed markers
    ax_anim.scatter(x[0], y[0], z[0], color='green', s=100, label='Start', marker='o')
    ax_anim.scatter(x[-1], y[-1], z[-1], color='red', s=100, label='End', marker='s')
    ax_anim.scatter(state[-1, 6], state[-1, 7], state[-1, 8], color='pink', s=100, label='Button', marker='*')
    ax_anim.legend()
    
    return ax_anim,

# Create animation
anim = FuncAnimation(fig_anim, update_frame, frames=len(x), interval=100, blit=False, repeat=True)

# Display animation
HTML(anim.to_jshtml())